In [1]:
#Load packages

import pandas as pd
import numpy as np

from pathlib import Path
import json
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

In [6]:
# Reuse / Define Core Helper Functions

def ks_statistic(y_true, y_score):
    temp = pd.DataFrame({
        "y_true": y_true,
        "y_score": y_score
    }).sort_values("y_score", ascending=False)

    temp["good"] = (temp["y_true"] == 0).astype(int)
    temp["bad"] = (temp["y_true"] == 1).astype(int)

    temp["cum_good"] = temp["good"].cumsum() / temp["good"].sum()
    temp["cum_bad"] = temp["bad"].cumsum() / temp["bad"].sum()

    return (temp["cum_bad"] - temp["cum_good"]).abs().max()


def calculate_classification_diagnostics(
    y_train,
    p_train,
    y_val,
    p_val,
    threshold=0.50
):
    rows = []

    for dataset_name, y_true, p_score in [
        ("train", y_train, p_train),
        ("validation", y_val, p_val)
    ]:

        y_pred = (p_score >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        rows.append({
            "dataset": dataset_name,
            "threshold": threshold,
            "auc": roc_auc_score(y_true, p_score),
            "gini": 2 * roc_auc_score(y_true, p_score) - 1,
            "ks": ks_statistic(y_true, p_score),
            "pr_auc": average_precision_score(y_true, p_score),
            "log_loss": log_loss(y_true, p_score),
            "brier_score": brier_score_loss(y_true, p_score),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp
        })

    return pd.DataFrame(rows)

print("Diagnostic helpers loaded.")


def get_next_model_number(model_registry, prefix):
    existing_numbers = []

    for model_num in model_registry.keys():
        if model_num.startswith(prefix):
            number_part = model_num.replace(prefix, "")
            if number_part.isdigit():
                existing_numbers.append(int(number_part))

    return max(existing_numbers) + 1 if existing_numbers else 1


def run_nb_model(
    model_registry,
    model,
    model_number,
    model_name,
    X_train,
    X_val,
    y_train,
    y_val,
    features,
    threshold=0.50,
    analyst_comments="",
    run_permutation=True,
    permutation_scoring="roc_auc",
    permutation_repeats=5,
    display_outputs=True
):

    if model_number in model_registry:
        raise ValueError(f"{model_number} already exists.")

    features = list(features)

    model.fit(X_train[features], y_train)

    p_train = model.predict_proba(X_train[features])[:, 1]
    p_val = model.predict_proba(X_val[features])[:, 1]

    diagnostics_table = calculate_classification_diagnostics(
        y_train=y_train,
        p_train=p_train,
        y_val=y_val,
        p_val=p_val,
        threshold=threshold
    )

    if run_permutation:
        perm_result = permutation_importance(
            model,
            X_val[features],
            y_val,
            scoring=permutation_scoring,
            n_repeats=permutation_repeats,
            random_state=42,
            n_jobs=-1
        )

        permutation_importance_df = pd.DataFrame({
            "variable": features,
            "permutation_importance_mean": perm_result.importances_mean,
            "permutation_importance_std": perm_result.importances_std
        }).sort_values(
            "permutation_importance_mean",
            ascending=False
        ).reset_index(drop=True)
    else:
        permutation_importance_df = pd.DataFrame()

    metadata = {
        "model_number": model_number,
        "model_name": model_name,
        "model_class": type(model).__name__,
        "features": features,
        "num_features": len(features),
        "threshold": threshold,
        "parameters": model.get_params(),
        "analyst_comments": analyst_comments
    }

    model_registry[model_number] = {
        "metadata": metadata,
        "model": model,
        "diagnostics": diagnostics_table,
        "feature_importance": pd.DataFrame(),
        "permutation_importance": permutation_importance_df,
        "p_train": p_train,
        "p_val": p_val
    }

    if display_outputs:
        print("=" * 100)
        print(f"{model_number}: {model_name}")
        print("=" * 100)

        print("\nDIAGNOSTICS:")
        display(diagnostics_table)

        print("\nPERMUTATION IMPORTANCE:")
        display(permutation_importance_df.head(15))

        if analyst_comments:
            print("\nANALYST COMMENTS:")
            print(analyst_comments)

    return model_registry

def save_model_artifact(model_registry, model_id, model_dir, config_dir):
    record = model_registry[model_id]

    model_path = model_dir / f"{model_id}.joblib"
    config_path = config_dir / f"{model_id}_config.json"

    joblib.dump(record["model"], model_path)

    metadata = record.get("metadata", {}).copy()

    config = {
        "model_id": model_id,
        "model_class": str(type(record["model"]).__name__),
        "metadata": metadata,
        "features": metadata.get("features", None),
        "num_features": metadata.get("num_features", None),
    }

    if "diagnostics" in record:
        try:
            config["diagnostics_preview"] = record["diagnostics"].to_dict(orient="records")
        except Exception:
            pass

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4, default=str)

Diagnostic helpers loaded.


In [2]:
# Paths

CURRENT = Path.cwd()

if CURRENT.name == "notebooks":
    PROJECT_ROOT = CURRENT.parent
else:
    PROJECT_ROOT = CURRENT

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR = OUTPUT_DIR / "saved_models"
CONFIG_DIR = OUTPUT_DIR / "model_configs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)
print("Output Directory:", OUTPUT_DIR)

Project Root: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab
Data Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/data
Output Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs


In [3]:
# Load WOE Model-Ready Dataset

df_woe = pd.read_parquet(DATA_DIR / "df_logit_woe_ready.parquet")

print("WOE dataset shape:", df_woe.shape)
display(df_woe.head())

WOE dataset shape: (149390, 12)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines_woe,age_woe,MonthlyIncome_median_woe,DebtRatio_high_flag_woe,NumberOfDependents_median_woe,NumberOfDependents_missing_flag_woe,NumberRealEstateLoansOrLines_woe,NumberOfTime30-59DaysPastDueNotWorse_woe,NumberOfTime60-89DaysPastDueNotWorse_woe,NumberOfTimes90DaysLate_woe,MonthlyIncome_missing_flag_woe
0,1,-1.142022,-0.241241,0.144680,-0.042506,-0.206972,-0.008547,-0.251481,-1.614254,0.287551,0.389069,-0.039713
1,0,-1.142022,-0.241241,-0.273761,-0.042506,-0.100164,-0.008547,-0.241293,0.540919,0.287551,0.389069,-0.039713
2,0,-0.450577,-0.499303,-0.273761,-0.042506,0.149025,-0.008547,-0.241293,-0.901164,0.287551,-1.957686,-0.039713
3,0,0.699714,-0.499303,-0.273761,-0.042506,0.149025,-0.008547,-0.241293,0.540919,0.287551,0.389069,-0.039713
4,0,-1.142022,-0.241241,0.462746,-0.042506,0.149025,-0.008547,0.233164,-0.901164,0.287551,0.389069,-0.039713


In [4]:
# Define Target + Candidate Features

target = "SeriousDlqin2yrs"

candidate_features = [
    col for col in df_woe.columns
    if col != target
]

print("Target:", target)
print("Number of candidate features:", len(candidate_features))
print(candidate_features)

Target: SeriousDlqin2yrs
Number of candidate features: 11
['RevolvingUtilizationOfUnsecuredLines_woe', 'age_woe', 'MonthlyIncome_median_woe', 'DebtRatio_high_flag_woe', 'NumberOfDependents_median_woe', 'NumberOfDependents_missing_flag_woe', 'NumberRealEstateLoansOrLines_woe', 'NumberOfTime30-59DaysPastDueNotWorse_woe', 'NumberOfTime60-89DaysPastDueNotWorse_woe', 'NumberOfTimes90DaysLate_woe', 'MonthlyIncome_missing_flag_woe']


In [5]:
# Train / Validation Split

X = df_woe[candidate_features].copy()
y = df_woe[target].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("Train bad rate:", y_train.mean())
print("Val bad rate  :", y_val.mean())

X_train: (104573, 11)
X_val  : (44817, 11)
Train bad rate: 0.06699626098514913
Val bad rate  : 0.06700582368297744


In [8]:
# Initiate registry and model
wgnb_registry = {}

wgnb_model = GaussianNB()


In [9]:
# WGNB001 - Gaussian Naive Bayes Baseline all WOE Features

wgnb_registry = run_nb_model(
    model_registry=wgnb_registry,
    model=wgnb_model,
    model_number="WGNB001",
    model_name="Gaussian Naive Bayes - All WOE Features",
    X_train=X_train,
    X_val=X_val,
    y_train=y_train,
    y_val=y_val,
    features=candidate_features,
    analyst_comments=(
        "Baseline Gaussian Naive Bayes using Weight of Evidence transformed "
        "features. WOE variables are tested as smoother, risk-ordered numeric "
        "inputs that may better align with GaussianNB assumptions than raw variables."
    ),
    run_permutation=True,
    permutation_repeats=5,
    display_outputs=True
)

WGNB001: Gaussian Naive Bayes - All WOE Features

DIAGNOSTICS:


,dataset,threshold,auc,gini,ks,pr_auc,log_loss,brier_score,accuracy,precision,recall,f1,true_negative,false_positive,false_negative,true_positive
0,train,0.5,0.845697,0.691394,0.547286,0.368951,0.827928,0.092825,0.901170,0.345148,0.529546,0.417910,90528,7039,3296,3710
1,validation,0.5,0.850856,0.701712,0.557232,0.381327,0.820473,0.092742,0.901198,0.345679,0.531469,0.418898,38793,3021,1407,1596



PERMUTATION IMPORTANCE:


,variable,permutation_importance_mean,permutation_importance_std
0,RevolvingUtilizationOfUnsecuredLines_woe,0.049112,0.001737
1,NumberOfTimes90DaysLate_woe,0.043471,0.001059
2,NumberOfTime30-59DaysPastDueNotWorse_woe,0.036778,0.000875
3,NumberOfTime60-89DaysPastDueNotWorse_woe,0.027499,0.001322
4,age_woe,0.011624,0.000840
5,NumberOfDependents_missing_flag_woe,0.003061,0.001458
6,NumberRealEstateLoansOrLines_woe,0.001857,0.000320
7,MonthlyIncome_median_woe,0.001471,0.000341
8,NumberOfDependents_median_woe,0.000188,0.000325
9,DebtRatio_high_flag_woe,-0.000389,0.000281



ANALYST COMMENTS:
Baseline Gaussian Naive Bayes using Weight of Evidence transformed features. WOE variables are tested as smoother, risk-ordered numeric inputs that may better align with GaussianNB assumptions than raw variables.


Weight of Evidence transformation substantially improved Gaussian Naive Bayes by converting irregular raw credit variables into smoother risk-ordered numeric predictors.

In [10]:
# WGNB var_smoothing Grid

var_smoothing_grid = [
    1e-12,
    1e-11,
    1e-10,
    1e-9,
    1e-8,
    1e-7,
    1e-6,
    1e-5,
    1e-4
]

start_num = get_next_model_number(wgnb_registry, "WGNB")

for i, var_smoothing in enumerate(var_smoothing_grid, start=start_num):

    model_number = f"WGNB{i:03d}"

    wgnb_model = GaussianNB(
        var_smoothing=var_smoothing
    )

    wgnb_registry = run_nb_model(
        model_registry=wgnb_registry,
        model=wgnb_model,
        model_number=model_number,
        model_name=f"Gaussian Naive Bayes - WOE var_smoothing={var_smoothing}",
        X_train=X_train,
        X_val=X_val,
        y_train=y_train,
        y_val=y_val,
        features=candidate_features,
        analyst_comments=(
            "GaussianNB on WOE-transformed features. "
            f"Testing var_smoothing={var_smoothing} to evaluate probability stability and calibration."
        ),
        run_permutation=True,
        permutation_repeats=5,
        display_outputs=False
    )

print("WOE GaussianNB var_smoothing grid complete.")

WOE GaussianNB var_smoothing grid complete.


In [11]:
# WOE GaussianNB Summary Table

wgnb_summary_rows = []

for model_num, model_data in wgnb_registry.items():

    meta = model_data["metadata"]
    diag = model_data["diagnostics"]
    params = meta.get("parameters", {})

    train_row = diag.loc[diag["dataset"] == "train"].iloc[0]
    val_row = diag.loc[diag["dataset"] == "validation"].iloc[0]

    wgnb_summary_rows.append({
        "model_number": model_num,
        "model_name": meta["model_name"],
        "var_smoothing": params.get("var_smoothing"),
        "num_features": meta.get("num_features"),

        "train_auc": train_row["auc"],
        "val_auc": val_row["auc"],
        "auc_gap": train_row["auc"] - val_row["auc"],

        "train_ks": train_row["ks"],
        "val_ks": val_row["ks"],
        "ks_gap": train_row["ks"] - val_row["ks"],

        "val_pr_auc": val_row["pr_auc"],
        "val_brier": val_row["brier_score"],
        "val_log_loss": val_row["log_loss"],

        "val_accuracy": val_row["accuracy"],
        "val_precision": val_row["precision"],
        "val_recall": val_row["recall"],
        "val_f1": val_row["f1"]
    })

wgnb_summary_df = (
    pd.DataFrame(wgnb_summary_rows)
    .sort_values(
        by=["val_auc", "val_ks", "val_pr_auc"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(wgnb_summary_df)

,model_number,model_name,var_smoothing,num_features,train_auc,val_auc,auc_gap,train_ks,val_ks,ks_gap,val_pr_auc,val_brier,val_log_loss,val_accuracy,val_precision,val_recall,val_f1
0,WGNB010,Gaussian Naive Bayes - WOE var_smoothing=0.0001,1.000000e-04,11,0.845927,0.851095,-0.005168,0.547422,0.557399,-0.009977,0.381490,0.092718,0.819554,0.901176,0.345604,0.531469,0.418843
1,WGNB009,Gaussian Naive Bayes - WOE var_smoothing=1e-05,1.000000e-05,11,0.845723,0.850886,-0.005164,0.547300,0.557232,-0.009932,0.381354,0.092739,0.820380,0.901176,0.345604,0.531469,0.418843
2,WGNB008,Gaussian Naive Bayes - WOE var_smoothing=1e-06,1.000000e-06,11,0.845698,0.850858,-0.005160,0.547279,0.557232,-0.009952,0.381329,0.092741,0.820463,0.901198,0.345679,0.531469,0.418898
3,WGNB007,Gaussian Naive Bayes - WOE var_smoothing=1e-07,1.000000e-07,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381328,0.092742,0.820472,0.901198,0.345679,0.531469,0.418898
4,WGNB006,Gaussian Naive Bayes - WOE var_smoothing=1e-08,1.000000e-08,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381328,0.092742,0.820473,0.901198,0.345679,0.531469,0.418898
5,WGNB001,Gaussian Naive Bayes - All WOE Features,1.000000e-09,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381327,0.092742,0.820473,0.901198,0.345679,0.531469,0.418898
6,WGNB002,Gaussian Naive Bayes - WOE var_smoothing=1e-12,1.000000e-12,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381327,0.092742,0.820473,0.901198,0.345679,0.531469,0.418898
7,WGNB003,Gaussian Naive Bayes - WOE var_smoothing=1e-11,1.000000e-11,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381327,0.092742,0.820473,0.901198,0.345679,0.531469,0.418898
8,WGNB004,Gaussian Naive Bayes - WOE var_smoothing=1e-10,1.000000e-10,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381327,0.092742,0.820473,0.901198,0.345679,0.531469,0.418898
9,WGNB005,Gaussian Naive Bayes - WOE var_smoothing=1e-09,1.000000e-09,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381327,0.092742,0.820473,0.901198,0.345679,0.531469,0.418898


Once variables were transformed into Weight of Evidence form, Gaussian Naive Bayes became highly stable across variance smoothing settings, indicating that preprocessing quality dominated parameter sensitivity.

In [12]:
# WOE GaussianNB Top-N Pruning Grid - Use Baseline Permutation Ranking

ranked_vars = (
    wgnb_registry["WGNB010"]["permutation_importance"]
    ["variable"]
    .tolist()
)

top_n_grid = [3, 5, 8, 10]

start_num = get_next_model_number(wgnb_registry, "WGNB")

for i, top_n in enumerate(top_n_grid, start=start_num):

    model_number = f"WGNB{i:03d}"

    selected_features = ranked_vars[:top_n]

    wgnb_model = GaussianNB(
        var_smoothing=1e-4
    )

    wgnb_registry = run_nb_model(
        model_registry=wgnb_registry,
        model=wgnb_model,
        model_number=model_number,
        model_name=f"Gaussian Naive Bayes - WOE Top {top_n} Variables",
        X_train=X_train,
        X_val=X_val,
        y_train=y_train,
        y_val=y_val,
        features=selected_features,
        analyst_comments=(
            f"WOE GaussianNB using top {top_n} variables ranked by "
            "permutation importance."
        ),
        run_permutation=True,
        permutation_repeats=5,
        display_outputs=False
    )

print("WOE GaussianNB pruning grid complete.")

WOE GaussianNB pruning grid complete.


In [13]:
# WOE GaussianNB Final Summary Table

wgnb_summary_rows = []

for model_num, model_data in wgnb_registry.items():

    meta = model_data["metadata"]
    diag = model_data["diagnostics"]
    params = meta.get("parameters", {})

    train_row = diag.loc[diag["dataset"] == "train"].iloc[0]
    val_row = diag.loc[diag["dataset"] == "validation"].iloc[0]

    wgnb_summary_rows.append({
        "model_number": model_num,
        "model_name": meta["model_name"],
        "var_smoothing": params.get("var_smoothing"),
        "num_features": meta.get("num_features"),

        "train_auc": train_row["auc"],
        "val_auc": val_row["auc"],
        "auc_gap": train_row["auc"] - val_row["auc"],

        "train_ks": train_row["ks"],
        "val_ks": val_row["ks"],
        "ks_gap": train_row["ks"] - val_row["ks"],

        "val_pr_auc": val_row["pr_auc"],
        "val_brier": val_row["brier_score"],
        "val_log_loss": val_row["log_loss"],

        "val_accuracy": val_row["accuracy"],
        "val_precision": val_row["precision"],
        "val_recall": val_row["recall"],
        "val_f1": val_row["f1"]
    })

wgnb_summary_df = (
    pd.DataFrame(wgnb_summary_rows)
    .sort_values(
        by=["val_auc", "val_ks", "val_pr_auc"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(wgnb_summary_df)

,model_number,model_name,var_smoothing,num_features,train_auc,val_auc,auc_gap,train_ks,val_ks,ks_gap,val_pr_auc,val_brier,val_log_loss,val_accuracy,val_precision,val_recall,val_f1
0,WGNB012,Gaussian Naive Bayes - WOE Top 5 Variables,1.000000e-04,5,0.851010,0.854839,-0.003828,0.551902,0.568964,-0.017062,0.382216,0.092796,0.808771,0.900908,0.343831,0.527140,0.416196
1,WGNB013,Gaussian Naive Bayes - WOE Top 8 Variables,1.000000e-04,8,0.847920,0.853402,-0.005482,0.551497,0.565878,-0.014381,0.383217,0.092451,0.815938,0.901421,0.345016,0.524476,0.416226
2,WGNB014,Gaussian Naive Bayes - WOE Top 10 Variables,1.000000e-04,10,0.847316,0.852505,-0.005189,0.550517,0.562073,-0.011556,0.382884,0.092616,0.817952,0.901243,0.345158,0.528139,0.417478
3,WGNB010,Gaussian Naive Bayes - WOE var_smoothing=0.0001,1.000000e-04,11,0.845927,0.851095,-0.005168,0.547422,0.557399,-0.009977,0.381490,0.092718,0.819554,0.901176,0.345604,0.531469,0.418843
4,WGNB009,Gaussian Naive Bayes - WOE var_smoothing=1e-05,1.000000e-05,11,0.845723,0.850886,-0.005164,0.547300,0.557232,-0.009932,0.381354,0.092739,0.820380,0.901176,0.345604,0.531469,0.418843
5,WGNB008,Gaussian Naive Bayes - WOE var_smoothing=1e-06,1.000000e-06,11,0.845698,0.850858,-0.005160,0.547279,0.557232,-0.009952,0.381329,0.092741,0.820463,0.901198,0.345679,0.531469,0.418898
6,WGNB007,Gaussian Naive Bayes - WOE var_smoothing=1e-07,1.000000e-07,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381328,0.092742,0.820472,0.901198,0.345679,0.531469,0.418898
7,WGNB006,Gaussian Naive Bayes - WOE var_smoothing=1e-08,1.000000e-08,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381328,0.092742,0.820473,0.901198,0.345679,0.531469,0.418898
8,WGNB001,Gaussian Naive Bayes - All WOE Features,1.000000e-09,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381327,0.092742,0.820473,0.901198,0.345679,0.531469,0.418898
9,WGNB002,Gaussian Naive Bayes - WOE var_smoothing=1e-12,1.000000e-12,11,0.845697,0.850856,-0.005159,0.547286,0.557232,-0.009946,0.381327,0.092742,0.820473,0.901198,0.345679,0.531469,0.418898


Scaled GaussianNB improved with pruning but remained weakest.
BernoulliNB on one-hot bins became highly competitive.
WOE GaussianNB with top-5 pruning nearly matched BernoulliNB while producing strongest KS.

In [14]:
# Save WOE Gaussian Naive Bayes Champion Artifact

save_model_artifact(
    model_registry=wgnb_registry,
    model_id="WGNB012",
    model_dir=MODEL_DIR,
    config_dir=CONFIG_DIR
)

In [15]:
# Export WOE Gaussian Naive Bayes Champion Summary

wgnb_champion_id = "WGNB012"

wgnb_champion_path = OUTPUT_DIR / "04c3_nb_woe_champion_summary.xlsx"

with pd.ExcelWriter(wgnb_champion_path, engine="openpyxl") as writer:

    wgnb_summary_df.to_excel(
        writer,
        sheet_name="WGNB_Model_Comparison",
        index=False
    )

    wgnb_registry[wgnb_champion_id]["diagnostics"].to_excel(
        writer,
        sheet_name="WGNB012_Diagnostics",
        index=False
    )

    wgnb_registry[wgnb_champion_id]["permutation_importance"].to_excel(
        writer,
        sheet_name="WGNB012_Permutation",
        index=False
    )

print("Saved:", wgnb_champion_path)

Saved: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/04c3_nb_woe_champion_summary.xlsx


## WOE Gaussian Naive Bayes Summary

Weight of Evidence (WOE) transformed variables were tested with Gaussian Naive Bayes to evaluate whether scorecard-style feature engineering could better align with the model’s continuous probability assumptions than raw scaled variables. WOE variables convert each predictor into a monotonic, risk-ordered numeric signal based on observed good/bad behavior, often producing smoother and more stable inputs.

This representation materially improved Gaussian Naive Bayes performance relative to the earlier scaled-variable models. The result suggests that the weaker performance of the original GaussianNB models was driven less by the algorithm itself and more by the mismatch between raw variable distributions and Gaussian assumptions.

Variance smoothing was tested across multiple settings. Results were highly stable, indicating that once variables were expressed in WOE form, the model became relatively insensitive to hyperparameter tuning. This reinforces the idea that preprocessing quality mattered more than parameter optimization.

The strongest model came from supervised pruning rather than additional tuning. Using the top five WOE variables ranked by permutation importance, WGNB012 improved validation AUC and produced the strongest KS among all WOE GaussianNB models. This indicates that even within an already strong transformed feature space, removing weaker variables further improved signal concentration.

The final WOE Gaussian Naive Bayes champion was **WGNB012**.

### Practical Interpretation

- WOE transformation substantially improved GaussianNB.
- Hyperparameter tuning had limited incremental value.
- Feature pruning created the strongest final model.
- WOE GaussianNB nearly matched BernoulliNB performance while producing stronger KS separation.

### Final WOE NB Champion

**WGNB012** (Top 5 WOE Variables)